# EDA com download e exportação — League of Legends

Este notebook reproduz a análise de `arquivo-eda-original.ipynb` usando o dataset **League of Legends Diamond Ranked Games (10 min)**. Ele remove as células remanescentes do exemplo de Heart Disease, baixa o CSV público e exporta a versão tratada consumida pelo Streamlit.

- Fonte original: [Kaggle — League of Legends Diamond Ranked Games (10 min)](https://www.kaggle.com/datasets/bobbyscience/league-of-legends-diamond-ranked-games-10-min)
- Arquivo: `high_diamond_ranked_10min.csv`
- Unidade de análise: uma partida ranqueada Diamond/Master, observada aos 10 minutos.


## 1. Bibliotecas, caminhos e identidade visual

O notebook funciona tanto a partir da raiz do repositório quanto de `notebooks/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 5)

COR_NEUTRA = "#d9e2d0"
COR_CONTEXTO = "#f2b84b"
COR_DESTAQUE = "#176b4d"
COR_ALERTA = "#b45309"

RAIZ = Path("..") if Path("../data").exists() else Path(".")
PASTA_RAW = RAIZ / "data" / "raw"
PASTA_CLEAN = RAIZ / "data" / "clean"
PASTA_RAW.mkdir(parents=True, exist_ok=True)
PASTA_CLEAN.mkdir(parents=True, exist_ok=True)


## 2. Download do dataset

A URL abaixo é um espelho público do arquivo disponibilizado originalmente no Kaggle. O CSV bruto é salvo em `data/raw/` e continua ignorado pelo Git; somente a versão tratada será versionada.


In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/juanfelipehdezm/"
    "LOL-league-Of-Legends/master/high_diamond_ranked_10min.csv"
)

df_raw = pd.read_csv(DATA_URL)
df_raw.to_csv(PASTA_RAW / "dataset.csv", index=False)
print(f"Dataset bruto: {df_raw.shape[0]:,} partidas × {df_raw.shape[1]} colunas")
df_raw.head()


## 3. Inspeção estrutural

Verificamos dimensão, tipos, duplicatas e ausências antes de transformar os dados.


In [ ]:
display(df_raw.info())
print(f"Linhas duplicadas: {df_raw.duplicated().sum()}")
print(f"Valores ausentes: {int(df_raw.isna().sum().sum())}")
df_raw.describe().T


## 4. Tratamento e enriquecimento

A análise original não exige imputação: a base não possui ausências. Mantemos as variáveis observadas aos 10 minutos e adicionamos rótulos legíveis e medidas derivadas usadas nos dashboards.


In [ ]:
df = df_raw.drop_duplicates().copy()

df["team_label"] = df["blueWins"].map({0: "Time Vermelho", 1: "Time Azul"})
df["avg_level_band"] = pd.cut(
    df["blueAvgLevel"],
    bins=[-np.inf, 6.8, 7.0, 7.2, np.inf],
    labels=["Baixo", "Médio-baixo", "Médio-alto", "Alto"],
)
df["blueTotalGold_log"] = np.log1p(df["blueTotalGold"])
df["blueObjectives"] = (
    df["blueDragons"] + df["blueHeralds"] + df["blueTowersDestroyed"]
)
df["redObjectives"] = (
    df["redDragons"] + df["redHeralds"] + df["redTowersDestroyed"]
)

print(df[["blueWins", "team_label", "avg_level_band"]].head())
print(f"Base tratada: {df.shape[0]:,} partidas × {df.shape[1]} colunas")


## 5. Variáveis da EDA original

`blueWins` é o desfecho: `1` representa vitória do time azul e `0`, vitória do time vermelho. As variáveis quantitativas seguem a seleção feita no notebook original.


In [ ]:
VARS_QUANT = [
    "blueWardsPlaced", "blueWardsDestroyed", "blueKills", "blueDeaths",
    "blueAssists", "blueEliteMonsters", "blueTotalGold", "blueAvgLevel",
    "blueTotalExperience", "blueTotalMinionsKilled",
    "blueTotalJungleMinionsKilled", "blueGoldDiff", "blueExperienceDiff",
    "blueCSPerMin", "blueGoldPerMin", "redWardsPlaced",
    "redWardsDestroyed", "redKills", "redDeaths", "redAssists",
    "redEliteMonsters", "redTotalGold", "redAvgLevel",
    "redTotalExperience", "redTotalMinionsKilled",
    "redTotalJungleMinionsKilled", "redGoldDiff", "redExperienceDiff",
    "redCSPerMin", "redGoldPerMin",
]

VARS_QUALI = [
    "blueWins", "blueFirstBlood", "blueDragons", "blueHeralds",
    "blueTowersDestroyed", "redFirstBlood", "redDragons",
    "redHeralds", "redTowersDestroyed",
]

desc = df[VARS_QUANT].describe().T
desc["skewness"] = df[VARS_QUANT].skew()
desc["kurtosis"] = df[VARS_QUANT].kurtosis()
desc.round(2)


## 6. Distribuições por resultado

A comparação dá atenção a ouro, abates, experiência e objetivos — vantagens centrais da análise original aos dez minutos.


In [ ]:
variaveis_foco = [
    "blueTotalGold", "blueKills", "blueTotalExperience",
    "blueGoldDiff", "blueExperienceDiff", "blueObjectives",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, coluna in zip(axes.ravel(), variaveis_foco):
    sns.histplot(
        data=df, x=coluna, hue="team_label", kde=True, element="step",
        palette={"Time Vermelho": COR_CONTEXTO, "Time Azul": COR_DESTAQUE}, ax=ax,
    )
    ax.set_title(coluna)
fig.suptitle("Distribuições aos 10 minutos por vencedor", fontsize=15)
plt.tight_layout()
plt.show()


## 7. Correlação de Spearman

Spearman é mantida por coerência com a EDA original e por não exigir normalidade das distribuições.


In [ ]:
corr_spearman = df[VARS_QUANT].corr(method="spearman")
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))

plt.figure(figsize=(20, 17))
sns.heatmap(
    corr_spearman, mask=mask, cmap="BrBG", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.2, cbar_kws={"label": "Correlação de Spearman"},
)
plt.title("Correlação entre variáveis quantitativas aos 10 minutos")
plt.tight_layout()
plt.show()


## 8. Relações com o resultado da partida

Comparamos as vantagens do time azul aos dez minutos com o resultado final. Isso preserva a pergunta central do notebook original: quais sinais do início da partida mais separam vitórias e derrotas?


In [ ]:
resumo_resultado = (
    df.groupby("team_label", observed=True)
    .agg(
        partidas=("gameId", "count"),
        ouro_azul_medio=("blueTotalGold", "mean"),
        diferenca_ouro_media=("blueGoldDiff", "mean"),
        diferenca_xp_media=("blueExperienceDiff", "mean"),
        abates_azuis_medios=("blueKills", "mean"),
        objetivos_azuis_medios=("blueObjectives", "mean"),
    )
    .round(2)
)
resumo_resultado


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=df, x="blueGoldDiff", y="blueExperienceDiff", hue="team_label",
    palette={"Time Vermelho": COR_CONTEXTO, "Time Azul": COR_DESTAQUE},
    alpha=0.45, ax=axes[0],
)
axes[0].axvline(0, color="#6b7280", linewidth=1)
axes[0].axhline(0, color="#6b7280", linewidth=1)
axes[0].set_title("Vantagem de ouro × vantagem de experiência")

prop_nivel = pd.crosstab(df["avg_level_band"], df["team_label"], normalize="index")
prop_nivel.plot(
    kind="bar", stacked=True, color=[COR_DESTAQUE, COR_CONTEXTO], ax=axes[1]
)
axes[1].set_title("Resultado por faixa de nível médio do time azul")
axes[1].set_ylabel("Proporção")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


## 9. Planejamento e validação das métricas do Dashboard 2

As métricas usam apenas informações disponíveis aos dez minutos e comunicam equilíbrio, vantagem econômica, combate e controle de objetivos.


In [ ]:
taxa_vitoria_azul = df["blueWins"].mean() * 100
diferenca_ouro_media = df["blueGoldDiff"].mean()
diferenca_abates_media = (df["blueKills"] - df["redKills"]).mean()
controle_objetivos_azul = (
    df["blueObjectives"] / (df["blueObjectives"] + df["redObjectives"]).replace(0, np.nan)
).mean() * 100

print(f"Vitórias do time azul: {taxa_vitoria_azul:.1f}%")
print(f"Diferença média de ouro (azul - vermelho): {diferenca_ouro_media:+.1f}")
print(f"Diferença média de abates (azul - vermelho): {diferenca_abates_media:+.2f}")
print(f"Participação azul nos objetivos disputados: {controle_objetivos_azul:.1f}%")


In [ ]:
resumo_faixa = (
    df.groupby("avg_level_band", observed=True, as_index=False)
    .agg(
        partidas=("gameId", "count"),
        taxa_vitoria_azul=("blueWins", lambda serie: serie.mean() * 100),
        diferenca_ouro_media=("blueGoldDiff", "mean"),
        diferenca_xp_media=("blueExperienceDiff", "mean"),
    )
)
resumo_faixa.round(2)


## 10. Exportação

A versão tratada é a fonte única do `app.py`. O arquivo mantém todas as colunas originais e acrescenta apenas os campos derivados documentados acima.


In [ ]:
df.to_csv(PASTA_CLEAN / "dataset.csv", index=False)

print(f"Raw salvo em:   {PASTA_RAW / 'dataset.csv'} — {df_raw.shape}")
print(f"Clean salvo em: {PASTA_CLEAN / 'dataset.csv'} — {df.shape}")
print("Campos derivados:", [
    "team_label", "avg_level_band", "blueTotalGold_log",
    "blueObjectives", "redObjectives",
])


## Síntese

1. A base reúne partidas ranqueadas de alto nível observadas aos dez minutos, sem valores ausentes.
2. Diferenças de ouro e experiência representam a vantagem relativa entre os times e são mais informativas que totais isolados.
3. Abates, experiência, ouro e objetivos permitem comparar domínio de combate, crescimento econômico e controle de mapa.
4. A variável `blueWins` conecta os sinais iniciais ao vencedor final sem transformar associação em causalidade.
5. O dashboard deve permitir filtrar cenários e comparar as distribuições dos dois vencedores, mantendo as limitações observacionais do dataset.
